# Parsa — Margin-Mean-Variance lossCovers **only** Parsa's section of `task.pdf`. Yasi's environment/baseline work is aseparate notebook (`yasi_environment_and_baselines.ipynb`); the two share nothing butthe repo.### Checklist- [ ] locate and read the loss-function section of the paper- [ ] write out the exact MMV formula (margin term + how mu-hat / variance feed in)- [ ] confirm how censored vs. uncensored samples are treated differently- [ ] `MMVLoss` following SAT's `Loss` base-class conventions- [ ] `conf/tasks/losses/mmv.yaml`- [ ] unit test following `tests/loss/` patterns- [ ] verify on a toy example in plain PyTorch/NumPy- [ ] MMV swapped in on `SurvivalTaskHead`, run on METABRIC- [ ] compared against the baseline `nllpch` numbers**Which paper.** The DOI in `task.pdf` (`10.1007/s10994-024-06686-w`) resolves to*UniSurv: Adaptive Transformer Modelling of Density Function for Nonparametric SurvivalAnalysis*, Machine Learning (2024) — [arXiv:2409.06209](https://arxiv.org/abs/2409.06209).It is not titled after MMV, which is why it is hard to find from the name alone.

## Options

In [ ]:
# survtrace_metabric = standardized features + SurvTRACE's model/optimiser + the
# tau-truncated IPCW C-td metric, so these numbers are comparable with published
# tables. See SCOPE.md.
DATASET = "survtrace_metabric"

# lambda_v. The paper grids {0.001, 0.01, 0.1, 1}, but lambda_v = 1 was
# catastrophic here (Brier 0.50 = coin flip), so it is left out by default.
MMV_VARIANCE_WEIGHTS = [0.001, 0.01, 0.1]
COMPARE_AGAINST      = "nllpch"   # the baseline recipe to beat

RUN_TRAINING = True     # False = maths + tests only, no pipeline
SMOKE_TEST   = True     # 3 epochs; proves it trains before you commit to a full run
SMOKE_EPOCHS = 3
NUM_EPOCHS   = None

SEEDS = [0]
# SEEDS = [0, 1, 2, 3, 4]   # <- uncomment for error bars; required before claiming
#                              MMV beats the baseline. A single seed cannot produce a
#                              spread: the sd inside metrics.json is computed within one
#                              run and is always exactly 0.0.

EXTRA_OVERRIDES = []

INSTALL_DEPS = True
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # cloned automatically

# Kaggle's "GPU T4 x2" has 2 GPUs, which makes HF Trainer use DataParallel and
# crash. Keep this True unless you know you want multi-GPU.
USE_SINGLE_GPU   = True
SINGLE_GPU_INDEX = "0"

RESULTS_DIR  = "/kaggle/working/results_mmv"

## 1 · Setup

In [ ]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = None

def looks_like_repo(p: Path) -> bool:
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

# 1. already here from an earlier cell run in this session
if looks_like_repo(WORK / "dmmst"):
    REPO = WORK / "dmmst"
    print("Using existing clone.")

# 2. clone it - this is the normal path, no Kaggle Dataset needed
elif GIT_URL:
    dst = WORK / "dmmst"
    print(f"Cloning {GIT_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(dst)], check=True)
    REPO = dst

# 3. fallback: someone attached the folder as a Kaggle Dataset instead
if REPO is None:
    for base in [Path("/kaggle/input"), Path.cwd()]:
        if not base.exists():
            continue
        for cand in sorted(base.rglob("conf")):
            if looks_like_repo(cand.parent):
                dst = WORK / "dmmst"
                if not dst.exists():
                    # /kaggle/input is read-only and the pipeline writes into data/
                    print(f"Copying {cand.parent} -> {dst}")
                    shutil.copytree(cand.parent, dst)
                REPO = dst
                break
        if REPO:
            break

assert REPO is not None, (
    "Could not obtain the code. Set GIT_URL (default should just work), or attach "
    "the dmmst folder as a Kaggle Dataset."
)
REPO = REPO.resolve()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repo:", REPO)

# the datasets ship inside the repo, so nothing else to download
for d, f in [("metabric", "metabric_IHC4_clinical_train_test.h5"),
             ("hsa-synthetic", "simulated_data.csv"),
             ("support", "support_train_test.h5")]:
    p = REPO / "data" / d / f
    print(f"  data/{d:14} {'OK' if p.is_file() else 'MISSING'}")

In [ ]:
def sh(cmd, check=True):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

if INSTALL_DEPS:
    import torch
    tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    torchsurv_pin = "torchsurv" if tv >= (2, 8) else "torchsurv==0.1.4"
    print(f"torch {torch.__version__} -> {torchsurv_pin}")
    pkgs = [
        "transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22",
        "evaluate>=0.4.3", "accelerate>=1.4.0",
        "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
        torchsurv_pin,
        "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
        "numba", "nvidia-ml-py", "polars", "tqdm",
    ]

    # scikit-survival supplies the reference IPCW C-td / Brier used by the
    # published tables. Which release to use depends entirely on the numpy already
    # in the image, and NUMPY MUST NOT BE MOVED: pandas, pyarrow and friends are
    # compiled against it, and changing it under them gives
    #     ValueError: numpy.dtype size changed ... Expected 96, got 88
    # Pinning numpy to exactly what is installed makes pip fail loudly instead.
    import numpy as _np
    _numpy_version = _np.__version__
    _sksurv = ("scikit-survival" if int(_numpy_version.split(".")[0]) >= 2
               else "scikit-survival==0.22.2")
    print(f"numpy {_numpy_version} -> installing {_sksurv} (numpy held fixed)")
    pkgs += [_sksurv, f"numpy=={_numpy_version}"]
    sh("pip install -q " + " ".join(f'"{p}"' for p in pkgs))
    print("\nIf pip changed an already-imported package, restart the session "
          "(Run -> Restart) and re-run with INSTALL_DEPS = False.")
else:
    print("Skipping install.")

In [ ]:
import torch
from sat.loss import MMVLoss
print("MMVLoss imported OK ·", "cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
import re
from tqdm.auto import tqdm

ENV = os.environ.copy()
ENV["TOKENIZERS_PARALLELISM"] = "false"
ENV["HYDRA_FULL_ERROR"] = "1"
ENV["PYTHONUNBUFFERED"] = "1"      # without this the child buffers and the bar stalls

# Kaggle's "GPU T4 x2" exposes TWO GPUs. HF Trainer then does
#     if self.args.n_gpu > 1: model = nn.DataParallel(model)
# and DataParallel scatters every forward input across devices, which dies with
#     RuntimeError: chunk expects at least a 1-dimensional tensor
# on this model because some inputs are 0-dimensional.
#
# Pinning to one GPU is also the right call on merit: the model is ~3 MB and the
# datasets are small, so this is overhead-bound, and DataParallel's per-step
# scatter/gather would make it SLOWER even if it worked.
if USE_SINGLE_GPU:
    ENV["CUDA_VISIBLE_DEVICES"] = SINGLE_GPU_INDEX

import torch as _torch
if _torch.cuda.is_available():
    _n = _torch.cuda.device_count()
    print(f"{_n} GPU(s) visible to this notebook"
          + (f" -> pinning child runs to device {SINGLE_GPU_INDEX}" if USE_SINGLE_GPU and _n > 1
             else ""))

# lines worth surfacing live; everything else is kept for the failure tail only
_INTERESTING = re.compile(
    r"error|exception|traceback|warning|failed|"
    r"eval_ipcw_weighted_avg|eval_brier_weighted_avg|Save model|Write prediction",
    re.I,
)
_PROBLEM = re.compile(r"error|exception|traceback|failed|warning", re.I)
_EPOCH = re.compile(r"'epoch': ([0-9.]+)")


def expected_epochs():
    """Total epochs for the progress bar, or None if the config decides."""
    if SMOKE_TEST:
        return SMOKE_EPOCHS
    return NUM_EPOCHS      # None -> indeterminate bar (still shows count + elapsed)


def run_sat(script, experiment, overrides=(), label=None, tail_lines=15):
    """Run a sat entry point, streaming progress live.

    subprocess.run(stdout=PIPE) blocks until the process exits, which on a long
    run means staring at an empty cell for 20 minutes. This streams instead, and
    drives a tqdm bar off the trainer's own "'epoch': N" log lines.
    """
    cmd = [sys.executable, "-m", f"sat.{script}", f"experiments={experiment}", *overrides]
    label = label or f"{script} {experiment}"
    print()
    print("=" * 72)
    print(label)
    print("$ " + " ".join(cmd[2:]))
    print("=" * 72)

    total = expected_epochs() if script == "finetune" else None
    bar = tqdm(total=total, desc=label[:40], unit="ep", leave=False,
               bar_format="{l_bar}{bar}| {n:.0f}/{total_fmt} [{elapsed}<{remaining}]"
                          if total else "{desc}: {n:.0f} ep [{elapsed}]")

    t0 = time.time()
    lines = []
    last_shown = [0.0]     # throttle for routine progress lines
    proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, text=True, bufsize=1,
                            errors="replace",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        for raw in proc.stdout:
            # tqdm rewrites one line in place; keep only the final segment
            line = raw.rstrip(chr(10)).split(chr(13))[-1].rstrip()
            if not line:
                continue
            lines.append(line)

            m = _EPOCH.search(line)
            if m:
                ep = float(m.group(1))
                bar.n = min(ep, total) if total else ep
                bar.refresh()

            # Always surface problems. Throttle everything else: with
            # eval_steps=1 a 500-epoch run emits ~1500 metric lines, which would
            # bury the notebook.
            is_problem = _PROBLEM.search(line)
            now = time.time()
            if is_problem or (_INTERESTING.search(line) and now - last_shown[0] > 10):
                bar.write("   " + line[:160])
                if not is_problem:
                    last_shown[0] = now
    finally:
        proc.wait()
        bar.close()

    dt = time.time() - t0
    ok = proc.returncode == 0

    # keep the full log: the live view is throttled, and on failure this is the
    # only record of what happened
    try:
        safe = re.sub(r"[^A-Za-z0-9._-]+", "_", label).strip("_")[:80]
        logdir = Path(RESULTS_DIR) / "logs"
        logdir.mkdir(parents=True, exist_ok=True)
        (logdir / f"{'FAILED_' if not ok else ''}{safe}.log").write_text(
            chr(10).join(lines), encoding="utf8")
    except Exception as _e:
        print(f"  (could not save log: {_e})")

    if not ok:
        print("--- last lines ---")
        print(chr(10).join(lines[-tail_lines:]))

    swallowed = sum(1 for l in lines
                    if "Error in survival_predictions" in l or "Invalid predictions" in l)
    if swallowed:
        print(f"  !! {swallowed} swallowed prediction error(s) - metrics may be "
              f"hard-coded fallbacks rather than computed values")
    print(f"--> {'OK' if ok else 'FAILED'} in {dt:.1f}s")
    return ok, dt, chr(10).join(lines)


# `coxph` is the linear reference baseline and has its own entry point; every other
# model goes through sat.finetune.
def script_for(model):
    return "coxph" if model == "coxph" else "finetune"


def train_overrides(seed=0):
    """Common Hydra overrides.

    num_train_epochs lives at trainer.training_arguments.num_train_epochs - a bare
    `num_train_epochs=` override is rejected by Hydra. warmup_steps also gates
    eval_delay, so short runs need it at 0 or no evaluation happens and
    load_best_model_at_end fails.
    """
    ov = [f"seed={seed}"]
    if SMOKE_TEST:
        ov += [f"trainer.training_arguments.num_train_epochs={SMOKE_EPOCHS}",
               "warmup_steps=0"]
    elif NUM_EPOCHS:
        ov += [f"trainer.training_arguments.num_train_epochs={NUM_EPOCHS}"]
    return ov + list(EXTRA_OVERRIDES)

In [ ]:
import pandas as pd

# survtrace_metabric/* sets `dataset: metabric_numeric`, so metrics land under
# data/model-hub/metabric_numeric/. Map the experiment group to that directory.
_DATA_DIR = {"survtrace_metabric": "metabric_numeric",
             "survtrace_support": "support_numeric"}

def data_dir_for(experiment_group):
    return _DATA_DIR.get(experiment_group, experiment_group)


def collect_metrics(dataset, modelname, tag):
    """Copy a run's metrics.json into the shared results folder and parse it.

    metrics.json is NESTED:
        {"validation": {...}, "test": {"ipcw_weighted_avg": {"mean":..,"sd":..}, ...}}
    so a flat parse silently yields nothing.
    """
    src = REPO / "data" / "model-hub" / dataset / modelname / "metrics.json"
    if not src.is_file():
        print(f"  !! no metrics.json at {src}")
        return None
    dst = Path(RESULTS_DIR) / f"{tag}.json"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  saved -> {dst}")
    with open(dst) as f:
        return json.load(f)


def flatten(metrics, split="test"):
    """Pull {metric: mean} out of one split of the nested metrics.json."""
    out = {}
    for k, v in (metrics.get(split) or {}).items():
        if isinstance(v, dict) and "mean" in v:
            out[k] = v["mean"]
        elif isinstance(v, (int, float)):
            out[k] = v
    return out


# ctd_* come from the SurvTRACE-matched metric (tau-truncated, train IPCW) and are
# the ones comparable with published tables. ipcw_* are the older, non-comparable
# ones, kept so earlier runs still parse.
HEADLINE = ["ctd_weighted_avg", "ctd_0th_event_0.25", "ctd_0th_event_0.5",
            "ctd_0th_event_0.75", "brier_survtrace_weighted_avg",
            "mae_margin", "mae_uncensored",
            "ipcw_weighted_avg", "brier_weighted_avg", "loss",
            "within_subject_ipcw", "mismatch"]
NOISE = ("runtime", "samples_per_second", "steps_per_second", "n")

def results_table(results, title, split="test"):
    rows = []
    for tag, m in results.items():
        if m is None or m.get("_failed"):
            rows.append({"run": tag, "status": "FAILED"})
            continue
        flat = flatten(m, split)
        row = {"run": tag, "status": "ok"}
        for k in HEADLINE:
            if k in flat:
                row[k] = round(flat[k], 4)
        for k, v in flat.items():          # any remaining per-event metrics
            if k not in row and not k.endswith("_n") and k not in NOISE:
                row[k] = round(v, 4)
        rows.append(row)
    if not rows:
        print("no results yet")
        return None
    df = pd.DataFrame(rows).set_index("run").dropna(axis=1, how="all")
    print(f"\n### {title}  (split = {split})")
    display(df)
    return df


import re as _re

def aggregate_seeds(results, split="test"):
    """Mean +/- sd ACROSS seeds.

    The `variance`/`sd` fields inside a single metrics.json are computed within
    one run, so with one seed they are always exactly 0.0 - that is an artifact,
    not a real spread. Any spread worth reporting comes from re-running with
    different seeds, which is what this aggregates.
    """
    groups = {}
    for tag, m in results.items():
        if m is None or m.get("_failed"):
            continue
        base = _re.sub(r"_seed\d+$", "", tag)
        groups.setdefault(base, []).append(flatten(m, split))
    if not groups:
        print("nothing to aggregate")
        return None

    rows = []
    for base, runs in sorted(groups.items()):
        row = {"run": base, "seeds": len(runs)}
        keys = [k for k in runs[0]
                if not k.endswith("_n") and k not in NOISE]
        for k in HEADLINE + [k for k in keys if k not in HEADLINE]:
            vals = [r[k] for r in runs if k in r]
            if not vals:
                continue
            mean = sum(vals) / len(vals)
            if len(vals) > 1:
                sd = (sum((v - mean) ** 2 for v in vals) / (len(vals) - 1)) ** 0.5
                row[k] = f"{mean:.4f} +/- {sd:.4f}"
            else:
                row[k] = f"{mean:.4f}"
        rows.append(row)

    df = pd.DataFrame(rows).set_index("run")
    print(f"\n### Aggregated across seeds  (split = {split})")
    display(df)
    if all(r["seeds"] == 1 for r in rows):
        print("\nOnly one seed per configuration, so there is no spread to report.\n"
              "Uncomment the multi-seed line in Options before quoting any of these\n"
              "numbers in the paper.")
    return df

## 2 · The formula**Mean lifetime from the predicted density (Eq. 3)**$$\hat{\mu} = \sum_{t=T_0}^{T_{max}} \hat{S}(t)$$**Variance of that density about its mean (Eq. 4)**$$v = \sum_{t=T_0}^{T_{max}} \hat{p}_t \, (t - \hat{\mu})^2$$**Margin ("best guess") time for a censored subject (Eq. 5)** — from the populationKaplan–Meier estimator fitted on the *training* split:$$e_m^i = T^i + \frac{\int_{T^i}^{T_{max}} S_{km}(t)\,dt}{S_{km}(T^i)}$$**Margin-mean loss (Eq. 6)** — this is where censored and uncensored diverge:$$\mathcal{L}_{mm} = \frac{1}{2}\sum_{i=1}^{N}\Big[\underbrace{\delta^i\,(\hat{\mu}^i - T^i)^2}_{\text{uncensored: against the true time}} \;+\; \underbrace{(1-\delta^i)\,\omega^i\,(\hat{\mu}^i - e_m^i)^2}_{\text{censored: against the margin time}}\Big]$$$$\omega^i = 1 - S_{km}(T^i)$$### Censored vs. uncensored, precisely| | target | weight ||---|---|---|| uncensored ($\delta=1$) | observed time $T^i$ | 1 || censored ($\delta=0$) | margin time $e_m^i \ge T^i$ | $\omega^i = 1 - S_{km}(T^i)$ |Censored subjects are **not** dropped and **not** treated as events. They are pulledtoward a KM-derived best guess, down-weighted by $\omega$. Since $S_{km}$ decreases,$\omega$ is *small for early censoring* — exactly where the best guess is leasttrustworthy — and approaches 1 for late censoring.### How it maps onto this codebaseThe full objective is $\mathcal{L}_{total}=\mathcal{L}_s+\lambda_m\mathcal{L}_{mm}+\lambda_v\mathcal{L}_v+\lambda_d\mathcal{L}_d$ (Eq. 10).Only the middle two terms are MMV-specific, so only those are in `MMVLoss`; the othersalready exist and compose through `MetaLoss`:| Paper term | Here ||---|---|| $\mathcal{L}_s$ | `SATNLLPCHazardLoss` || $\mathcal{L}_{mm}+\lambda_v\mathcal{L}_v$ | **`MMVLoss`** ← new || $\mathcal{L}_d$ | `SampleRankingLoss` |$\lambda_m$ is therefore the `coeffs` entry given to `MMVLoss` inside `MetaLoss`.**Worth flagging:** Eq. 5 is the *same* best-guess construction as Eq. 6 of our ownpaper, and was already implemented as `sat.utils.km.KaplanMeierArea.best_guess`.`MMVLoss` reuses it rather than reimplementing — §3 checks they agree exactly.

## 3 · Toy verification in plain NumPy

In [ ]:
# MMVLoss vs. a from-scratch NumPy implementation of Eqs. 3-6. No pipeline, no training.
import numpy as np, torch, pandas as pd, tempfile
from sat.loss import MMVLoss
from sat.models.heads import SAOutput
from sat.utils.km import KaplanMeierArea

NUM_EVENTS, MAX_TIME, LAMBDA_V = 1, 12.0, 0.1
CUTS = np.array([2.0, 4.0, 6.0, 8.0, 10.0])

tmp = Path(tempfile.mkdtemp())
pd.DataFrame({"cuts": CUTS}).to_csv(tmp / "cuts.csv", index=False, header=False)
rng = np.random.default_rng(0)
train = pd.DataFrame({"duration_event1": rng.uniform(1, 10, 300),
                      "event1": rng.integers(0, 2, 300)})
train.to_csv(tmp / "train.csv", index=False)

loss_fn = MMVLoss(duration_cuts=str(tmp / "cuts.csv"), training_set=str(tmp / "train.csv"),
                  num_events=NUM_EVENTS, max_time=MAX_TIME, variance_weight=LAMBDA_V)

# subject 0 observed at T=5; subject 1 censored at T=7
S         = np.array([[1.0, 0.85, 0.65, 0.45, 0.25, 0.10],
                      [1.0, 0.90, 0.75, 0.60, 0.40, 0.20]])
durations = np.array([[5.0], [7.0]])
events    = np.array([[1.0], [0.0]])

# ---- reference, straight from the equations ----
grid  = np.concatenate([[0.0], CUTS, [MAX_TIME]])
S_ext = np.concatenate([S, np.zeros((2, 1))], axis=1)
mu_np = np.sum(np.diff(grid) * (S_ext[:, :-1] + S_ext[:, 1:]) / 2, axis=1)      # Eq. 3
pdf   = S_ext[:, :-1] - S_ext[:, 1:]
mids  = (grid[:-1] + grid[1:]) / 2
v_np  = np.sum(pdf * (mids[None, :] - mu_np[:, None]) ** 2, axis=1)             # Eq. 4

km  = KaplanMeierArea(train["duration_event1"], train["event1"] == 1)
e_m = durations[:, 0].copy(); w = np.ones(2)
e_m[1] = km.best_guess(np.array([7.0]))[0]                                      # Eq. 5
w[1]   = 1.0 - km.predict(np.array([7.0]))[0]
delta  = events[:, 0]
l_mm   = 0.5 * (delta * (mu_np - durations[:, 0]) ** 2 +
                (1 - delta) * w * (mu_np - e_m) ** 2)                           # Eq. 6
expected = l_mm.mean() + LAMBDA_V * v_np.mean()

# ---- implementation under test ----
survival = torch.tensor(S, dtype=torch.float32).unsqueeze(1)
filler   = torch.zeros(2, NUM_EVENTS)
refs = torch.cat([filler, torch.tensor(events, dtype=torch.float32),
                  filler, torch.tensor(durations, dtype=torch.float32)], dim=1)
actual = loss_fn(SAOutput(loss=None, logits=None, survival=survival,
                          hazard=None, risk=None), refs).item()

print(f"  mu_hat      (Eq. 3)  {np.round(mu_np, 4)}")
print(f"  variance    (Eq. 4)  {np.round(v_np, 4)}")
print(f"  margin time (Eq. 5)  censored T=7.0 -> e_m={e_m[1]:.4f}, w={w[1]:.4f}")
print(f"  total       (Eq. 6)  expected={expected:.6f}  MMVLoss={actual:.6f}")
assert abs(expected - actual) < 1e-4
print("\n  MATCH - MMVLoss reproduces Eqs. 3-6 exactly.")

## 4 · Unit tests

In [ ]:
sh(f"{sys.executable} -m pytest tests/loss/survival/test_mmv.py -v "
   f"-p no:cacheprovider --no-header", check=False)

## 5 · MMV vs. `nllpch` on METABRICSame task head, same data, same seed — only the loss changes. Three recipes:| `tasks/losses=` | Objective ||---|---|| `nllpch` | $\mathcal{L}_{PCH}$ — the baseline || `mmv` | $\mathcal{L}_{mm}+\lambda_v\mathcal{L}_v$ — MMV alone || `nllpch_mmv` | $\mathcal{L}_{PCH}+\lambda_m(\mathcal{L}_{mm}+\lambda_v\mathcal{L}_v)$ — combined |`modelname=` is overridden every time; without it each variant would overwrite theprevious run's `metrics.json` and you would compare a run against itself.

In [ ]:
MMV_RESULTS = {}

if RUN_TRAINING:
    for step in ["prepare_data", "train_tokenizer", "train_labeltransform"]:
        ok, _, _ = run_sat(step, f"{DATASET}/survival")
        assert ok, f"{step} failed"

    DATA_DIR = data_dir_for(DATASET)

    for seed in SEEDS:
        tag = f"{COMPARE_AGAINST}_seed{seed}"
        ok, _, _ = run_sat("finetune", f"{DATASET}/survival",
                           train_overrides(seed) + [f"tasks/losses={COMPARE_AGAINST}",
                                                    "tasks/metrics=survtrace_mae",
                                                    f"modelname=survival_{COMPARE_AGAINST}"],
                           label=f"BASELINE · {COMPARE_AGAINST} (seed {seed})")
        MMV_RESULTS[tag] = collect_metrics(DATA_DIR, f"survival_{COMPARE_AGAINST}",
                                           tag) if ok else {"_failed": True}

        combos = [(r, lv) for r in ["mmv", "nllpch_mmv"] for lv in MMV_VARIANCE_WEIGHTS]
        outer = tqdm(combos, desc=f"MMV recipes (seed {seed})", unit="run")
        for recipe, lam_v in outer:
            outer.set_postfix_str(f"{recipe} lv={lam_v}")
            if True:
                name = f"survival_{recipe}_lv{lam_v}"
                tag  = f"{recipe}_lv{lam_v}_seed{seed}"
                ok, _, _ = run_sat("finetune", f"{DATASET}/survival",
                                   train_overrides(seed) + [f"tasks/losses={recipe}",
                                                            f"mmv_variance_weight={lam_v}",
                                                            f"modelname={name}"],
                                   label=f"MMV · {recipe} (lambda_v={lam_v}, seed {seed})")
                MMV_RESULTS[tag] = collect_metrics(DATA_DIR, name, tag) if ok else {"_failed": True}
else:
    print("RUN_TRAINING = False - maths and tests only")

In [ ]:
table = results_table(MMV_RESULTS, "MMV vs. baseline — METABRIC") if RUN_TRAINING else None
if RUN_TRAINING:
    aggregate_seeds(MMV_RESULTS)
if RUN_TRAINING and SMOKE_TEST:
    print("\nSMOKE_TEST is on: 3 epochs, so these numbers say nothing about whether MMV\n"
          "helps. They prove it trains. Set SMOKE_TEST = False, SEEDS = [0,1,2,3,4]\n"
          "before drawing any conclusion.")

In [ ]:
out = Path(RESULTS_DIR)
if MMV_RESULTS:
    with open(out / "summary.json", "w") as f:
        json.dump(MMV_RESULTS, f, indent=2)
    if table is not None:
        table.to_csv(out / "mmv_results.csv")

print("Commands used\n")
for seed in SEEDS:
    print(f"python -m sat.finetune experiments={DATASET}/survival "
          f"tasks/losses={COMPARE_AGAINST} modelname=survival_{COMPARE_AGAINST} seed={seed}")
    for recipe in ["mmv", "nllpch_mmv"]:
        for lam_v in MMV_VARIANCE_WEIGHTS:
            print(f"python -m sat.finetune experiments={DATASET}/survival "
                  f"tasks/losses={recipe} mmv_variance_weight={lam_v} "
                  f"modelname=survival_{recipe}_lv{lam_v} seed={seed}")
print("\nFiles:", *[p.name for p in sorted(out.glob('*'))])

## 6 · Download everything as one zip

In [ ]:
ZIP_NAME = "dmmst_parsa_mmv_results.zip"

import platform, zipfile

out = Path(RESULTS_DIR)
out.mkdir(parents=True, exist_ok=True)

# environment snapshot - worth having next to any number that goes in the paper
try:
    import torch, transformers, datasets
    env = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpus_visible": torch.cuda.device_count(),
        "gpu_name": (torch.cuda.get_device_name(0)
                     if torch.cuda.is_available() else None),
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "seeds": SEEDS,
        "smoke_test": SMOKE_TEST,
        "smoke_epochs": SMOKE_EPOCHS if SMOKE_TEST else None,
    }
    (out / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf8")
    print("environment.json written")
except Exception as e:
    print("could not write environment.json:", e)

# the fully-resolved Hydra config of each run, so any result can be traced back to
# the exact composed config that produced it
cfgdir = out / "resolved_configs"
cfgdir.mkdir(exist_ok=True)
n_cfg = 0
# hydra.run.dir is ${log_dir}/${task_name}/runs/<timestamp>, i.e. under logs/
for hydra_cfg in sorted((REPO / "logs").rglob("config.yaml")):
    if hydra_cfg.parent.name != ".hydra":
        continue
    run_dir = hydra_cfg.parent.parent                 # .../runs/<timestamp>
    job = run_dir.parent.parent.name                  # sat-finetune-<model>-<data>-job
    dest = cfgdir / f"{job}__{run_dir.name}.yaml"
    try:
        shutil.copy2(hydra_cfg, dest)
        n_cfg += 1
    except Exception:
        pass

ZIP_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ZIP_PATH = ZIP_DIR / ZIP_NAME
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

n_files = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(out.parent))
            n_files += 1

size_mb = ZIP_PATH.stat().st_size / 1024 ** 2
print()
print(f"Wrote {ZIP_PATH}")
print(f"  {n_files} files, {size_mb:.2f} MB ({n_cfg} resolved configs)")
print()
print("Contents:")
for f in sorted(out.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(out)}  ({f.stat().st_size / 1024:.1f} KB)")

# clickable link in the notebook; otherwise use the Output panel on the right
try:
    from IPython.display import FileLink, display as _d
    print()
    print("Download (or use the Output panel on the right):")
    try:
        link = str(ZIP_PATH.relative_to(Path.cwd()))
    except ValueError:
        link = str(ZIP_PATH)
    _d(FileLink(link))
except Exception:
    print("Find it in the Kaggle Output panel on the right.")

## Notes**Before concluding anything about MMV.** One seed at 3 epochs is not evidence. Set`SMOKE_TEST = False`, `SEEDS = [0,1,2,3,4]`, and sweep`MMV_VARIANCE_WEIGHTS = [0.001, 0.01, 0.1, 1]` as the paper does. Compare `nllpch`against `nllpch_mmv` at matched seeds.**Scale matters here.** `L_PCH` is a log-likelihood (~0.8 on METABRIC) while MMV is asquared error in *time units* (~2400, since METABRIC durations run to ~355). Under`fixed` balancing the MMV term outweighs the likelihood by three orders of magnitudeand the likelihood is effectively ignored, so `nllpch_mmv.yaml` defaults to`balancing: scale`. Measured at 3 epochs, seed 0:| recipe | ipcw | brier | loss ||---|---|---|---|| `nllpch` | 0.5188 | 0.2069 | 0.78 || `mmv` | 0.5292 | 0.2317 | 2434 || `nllpch_mmv` (fixed) | 0.5116 | 0.2344 | 2428 || `nllpch_mmv` (scale) | 0.5249 | 0.2318 | 2.18 |That shows the balancing behaves, not that MMV helps — 3 epochs, one seed.**What MMV should be expected to improve.** It optimises the *mean lifetime* of thepredicted density, so it targets time-to-event accuracy (MAE/MSE) more directly thanranking. A large C-index gain would be surprising; better calibration and regressionerror is the plausible win. Worth adding `tasks/metrics=l1` to measure that.**Still open:** `MMVLoss` is a first implementation and `task.pdf` asks for it to bereviewed together before finalising. Two choices in particular are worth a second pairof eyes — the discrete density is taken as $p_k = S_k - S_{k+1}$ on interval midpoints,and `max_time` defaults to the largest duration cut, which sets where $S$ is assumed toreach zero.